# Project Main: First API Downloads

This is the Day 1 project notebook for the first lesson.

The goal is small, but it still follows the real project shape:

1. Set up the Python environment.
2. Read the API key.
3. Define three project APIs.
4. Write one function to download API JSON.
5. Look briefly at the JSON.
6. Save all raw API results to CSV.
7. Convert the useful JSON section into normal table-shaped CSV files.
8. Read the CSV back to confirm it was saved.

The detailed practice notebooks are separate:

- `day_1_2JSON.ipynb` for JSON read/write practice.
- `day_1_3CSV.ipynb` for CSV read/write practice.

## 1. Environment

We only need a few tools for the first project loop.

In [14]:
import json
from datetime import datetime, timedelta, timezone
from pathlib import Path
from pprint import pprint

import pandas as pd
import requests

## 2. Project Folders

Project raw data goes to `day_1/data/raw/`.

The API key file lives in `day_1/shared_assets/api_key.txt`.

Practice files stay beside the practice notebooks.

In [15]:
%pwd

'd:\\SIT OneDrive\\OneDrive - Singapore Institute Of Technology\\project_IMDA\\teaching\\slides\\day_1\\01_ingestion_api_json'

In [16]:
SINGAPORE_TZ = timezone(timedelta(hours=8))


def find_day1_base_dir():
    current = Path.cwd().resolve()

    if (current / "day_1.pptx").exists():
        return current

    for parent in current.parents:
        if parent.name == "day_1":
            return parent

    return current


BASE_DIR = find_day1_base_dir()
RAW_DIR = BASE_DIR / "data" / "raw"
SHARED_ASSETS_DIR = BASE_DIR / "shared_assets"
API_KEY_PATH = SHARED_ASSETS_DIR / "api_key.txt"

RAW_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project raw data folder: {RAW_DIR}")
print(f"API key file: {API_KEY_PATH}")

Project raw data folder: D:\SIT OneDrive\OneDrive - Singapore Institute Of Technology\project_IMDA\teaching\slides\day_1\data\raw
API key file: D:\SIT OneDrive\OneDrive - Singapore Institute Of Technology\project_IMDA\teaching\slides\day_1\shared_assets\api_key.txt


## 3. Read the API Key

The key is stored in a text file.

Do not print the key value in class. Just check whether it was found.

In [17]:
def read_api_key():
    if not API_KEY_PATH.exists():
        return None

    for line in API_KEY_PATH.read_text(encoding="utf-8").splitlines():
        key = line.strip()
        if key:
            return key

    return None


API_KEY = read_api_key()

if API_KEY:
    print("API key found.")
else:
    print("No API key found. Some APIs may still work, but add one to shared_assets/api_key.txt if needed.")

API key found.


## 4. Define the Three Project APIs

For Day 1, the project collects:

1. Taxi availability
2. 2-hour weather forecast
3. Current rainfall observations

In [18]:
API_SOURCES = [
    {
        "api_name": "taxi_availability",
        "url": "https://api.data.gov.sg/v1/transport/taxi-availability",
    },
    {
        "api_name": "weather_forecast",
        "url": "https://api-open.data.gov.sg/v2/real-time/api/two-hr-forecast",
    },
    {
        "api_name": "rainfall",
        "url": "https://api-open.data.gov.sg/v2/real-time/api/rainfall",
    },
]

for source in API_SOURCES:
    print(source["api_name"], "->", source["url"])

taxi_availability -> https://api.data.gov.sg/v1/transport/taxi-availability
weather_forecast -> https://api-open.data.gov.sg/v2/real-time/api/two-hr-forecast
rainfall -> https://api-open.data.gov.sg/v2/real-time/api/rainfall


## 5. Download API JSON

This is the main function for today.

It calls one API, sends the API key if available, checks the status, and returns the JSON payload with simple metadata.

In [19]:
def request_headers():
    if API_KEY:
        return {"X-API-Key": API_KEY}
    return {}


def fetch_json(api_name, url):
    response = requests.get(url, headers=request_headers(), timeout=30)
    response.raise_for_status()

    return {
        "api_name": api_name,
        "requested_url": response.url,
        "status_code": response.status_code,
        "downloaded_at": datetime.now(SINGAPORE_TZ).isoformat(timespec="seconds"),
        "payload": response.json(),
    }

## 6. Run the Downloads

Run this cell once to download all three APIs.

In [20]:
api_results = []

for source in API_SOURCES:
    result = fetch_json(source["api_name"], source["url"])
    api_results.append(result)
    print(f"Downloaded {result['api_name']} with status {result['status_code']}")

Downloaded taxi_availability with status 200
Downloaded weather_forecast with status 200
Downloaded rainfall with status 200


## 7. Look Briefly at the JSON

Do not try to understand every detail yet.

For now, check the top-level keys and a small sample from each API.

In [21]:
for result in api_results:
    payload = result["payload"]

    print("=" * 60)
    print(result["api_name"])
    print("Top-level keys:")
    pprint(list(payload.keys()))

    if "data" in payload:
        print("Keys inside data:")
        pprint(list(payload["data"].keys()))
    elif "features" in payload:
        print("Number of features:", len(payload["features"]))
        if payload["features"]:
            print("First feature keys:")
            pprint(list(payload["features"][0].keys()))

taxi_availability
Top-level keys:
['type', 'crs', 'features']
Number of features: 1
First feature keys:
['type', 'geometry', 'properties']
weather_forecast
Top-level keys:
['code', 'data', 'errorMsg']
Keys inside data:
['area_metadata', 'items']
rainfall
Top-level keys:
['code', 'data', 'errorMsg']
Keys inside data:
['stations', 'readings', 'readingType', 'readingUnit']


## 8. Save All Raw API Results to CSV

Each run appends three rows to one CSV file: one row per API.

The raw JSON is saved as text so we can inspect or re-process it later.

In [22]:
raw_file = RAW_DIR / "api_raw_downloads.csv"

rows = []
for result in api_results:
    rows.append({
        "api_name": result["api_name"],
        "requested_url": result["requested_url"],
        "status_code": result["status_code"],
        "downloaded_at": result["downloaded_at"],
        "raw_json": json.dumps(result["payload"], ensure_ascii=False),
    })

raw_df = pd.DataFrame(rows)
write_header = not raw_file.exists()

raw_df.to_csv(raw_file, mode="a", header=write_header, index=False)

print(f"Appended {len(raw_df)} rows to {raw_file}")

Appended 3 rows to D:\SIT OneDrive\OneDrive - Singapore Institute Of Technology\project_IMDA\teaching\slides\day_1\data\raw\api_raw_downloads.csv


## 9. Turn the Useful JSON Data into Normal CSV

The previous CSV is useful as a raw download log, but it is not the CSV most people imagine.

It has one big `raw_json` text cell per API result.

For analysis, we usually want rows and columns:

- For the newer real-time APIs, the useful content is inside the top-level `data` key.
- For weather, `data` contains `area_metadata`, `items`, and `api_info`.
- For rainfall, `data` contains `stations`, `readings`, `readingType`, and `readingUnit`.
- The taxi API is an older GeoJSON-style response. It does not use `data`; its useful rows are inside `features[0]`.

First, look at a shortened version of the structure. Lists are trimmed so we can focus on the format, not the volume of data.

In [23]:
def preview_json_structure(value, max_list_items=2):
    if isinstance(value, dict):
        return {key: preview_json_structure(item, max_list_items) for key, item in value.items()}

    if isinstance(value, list):
        preview = [preview_json_structure(item, max_list_items) for item in value[:max_list_items]]
        if len(value) > max_list_items:
            preview.append(f"... {len(value) - max_list_items} more items")
        return preview

    return value


for result in api_results:
    payload = result["payload"]
    useful_part = payload["data"] if "data" in payload else payload

    print("=" * 60)
    print(result["api_name"])
    pprint(preview_json_structure(useful_part), width=100)

taxi_availability
{'crs': {'properties': {'href': 'http://spatialreference.org/ref/epsg/4326/ogcwkt/',
                        'type': 'ogcwkt'},
         'type': 'link'},
 'features': [{'geometry': {'coordinates': [[103.6142572761, 1.2556229087],
                                            [103.6142579466, 1.2559322942],
                                            '... 1984 more items'],
                            'type': 'MultiPoint'},
               'properties': {'api_info': {'status': 'healthy'},
                              'taxi_count': 1986,
                              'timestamp': '2026-06-20T09:10:14+08:00'},
               'type': 'Feature'}],
 'type': 'FeatureCollection'}
weather_forecast
{'area_metadata': [{'label_location': {'latitude': 1.375, 'longitude': 103.839},
                    'name': 'Ang Mo Kio'},
                   {'label_location': {'latitude': 1.321, 'longitude': 103.924}, 'name': 'Bedok'},
                   '... 45 more items'],
 'items': [{'forecasts

Now create one clean CSV per API.

The important idea is: choose the list that represents repeated observations, turn each item into one row, then use `df.to_csv(...)`.

In [24]:
# Turn an ISO datetime string into a safe filename timestamp.
# Example: 2026-06-07T22:22:21+08:00 -> 20260607_222221
def filename_timestamp(downloaded_at):
    return datetime.fromisoformat(downloaded_at).strftime("%Y%m%d_%H%M%S")


# Convert one downloaded API result into a normal table.
# A normal table means: one observation per row, one field per column.
def result_to_table(result):
    # These names come from our own fetch_json() function.
    # payload is the original JSON returned by the API.
    api_name = result["api_name"]
    payload = result["payload"]

    if api_name == "weather_forecast":
        # For this API, the useful information is inside payload["data"].
        data = payload["data"]

        # items is a list. The first item contains the current forecast block.
        item = data["items"][0]

        # area_metadata has location information for each area.
        # Build a lookup dictionary so we can find latitude/longitude by area name.
        area_lookup = {
            area["name"]: area["label_location"]
            for area in data["area_metadata"]
        }

        # Each forecast becomes one CSV row.
        rows = []
        for forecast in item["forecasts"]:
            location = area_lookup.get(forecast["area"], {})
            rows.append({
                "area": forecast["area"],
                "forecast": forecast["forecast"],
                "latitude": location.get("latitude"),
                "longitude": location.get("longitude"),
                "timestamp": item["timestamp"],
                "update_timestamp": item["update_timestamp"],
                "valid_start": item["valid_period"]["start"],
                "valid_end": item["valid_period"]["end"],
                "downloaded_at": result["downloaded_at"],
            })
        # Convert the list of row dictionaries into a DataFrame.
        return pd.DataFrame(rows)

    if api_name == "rainfall":
        # For rainfall, payload["data"] contains two important lists:
        # stations = station details; readings = rainfall values.
        data = payload["data"]

        # Build a lookup dictionary so stationId can tell us the station name/location.
        station_lookup = {station["id"]: station for station in data["stations"]}

        # The first reading block contains the latest set of station readings.
        reading_block = data["readings"][0]

        # Each station reading becomes one CSV row.
        rows = []
        for reading in reading_block["data"]:
            station = station_lookup.get(reading["stationId"], {})
            location = station.get("location", {})
            rows.append({
                "station_id": reading["stationId"],
                "station_name": station.get("name"),
                "value": reading["value"],
                "reading_unit": data["readingUnit"],
                "reading_type": data["readingType"],
                "latitude": location.get("latitude"),
                "longitude": location.get("longitude"),
                "timestamp": reading_block["timestamp"],
                "downloaded_at": result["downloaded_at"],
            })
        return pd.DataFrame(rows)

    if api_name == "taxi_availability":
        # Taxi availability is an older GeoJSON-style API.
        # It does not have payload["data"]. Its useful content is in features[0].
        feature = payload["features"][0]
        properties = feature["properties"]

        # coordinates is a long list of [longitude, latitude] taxi positions.
        # Each taxi coordinate becomes one CSV row.
        rows = []
        for taxi_index, coordinate in enumerate(feature["geometry"]["coordinates"], start=1):
            longitude, latitude = coordinate
            rows.append({
                "taxi_index": taxi_index,
                "longitude": longitude,
                "latitude": latitude,
                "timestamp": properties["timestamp"],
                "taxi_count": properties["taxi_count"],
                "downloaded_at": result["downloaded_at"],
            })
        return pd.DataFrame(rows)

    # Fallback: if we add another API later, let pandas flatten the JSON as best it can.
    return pd.json_normalize(payload.get("data", payload))


# Save one table-shaped CSV file for each API result.
table_files = []

for result in api_results:
    # Convert JSON into rows and columns.
    table_df = result_to_table(result)

    # Create a filename such as weather_forecast_20260607_222221.csv.
    table_file = RAW_DIR / f"{result['api_name']}_{filename_timestamp(result['downloaded_at'])}.csv"

    # Write the normal table to CSV.
    table_df.to_csv(table_file, index=False)
    table_files.append(table_file)
    print(f"Saved {len(table_df)} table rows to {table_file}")

Saved 1986 table rows to D:\SIT OneDrive\OneDrive - Singapore Institute Of Technology\project_IMDA\teaching\slides\day_1\data\raw\taxi_availability_20260620_091017.csv
Saved 47 table rows to D:\SIT OneDrive\OneDrive - Singapore Institute Of Technology\project_IMDA\teaching\slides\day_1\data\raw\weather_forecast_20260620_091018.csv
Saved 77 table rows to D:\SIT OneDrive\OneDrive - Singapore Institute Of Technology\project_IMDA\teaching\slides\day_1\data\raw\rainfall_20260620_091020.csv


Read one of the table-shaped CSV files back.

This is the kind of CSV we usually expect: one observation per row, one field per column.

In [25]:
example_table_df = pd.read_csv(table_files[0])

print(table_files[0])
example_table_df.head()

D:\SIT OneDrive\OneDrive - Singapore Institute Of Technology\project_IMDA\teaching\slides\day_1\data\raw\taxi_availability_20260620_091017.csv


,taxi_index,longitude,latitude,timestamp,taxi_count,downloaded_at
0,1,103.614257,1.255623,2026-06-20T09:10:14+08:00,1986,2026-06-20T09:10:17+08:00
1,2,103.614258,1.255932,2026-06-20T09:10:14+08:00,1986,2026-06-20T09:10:17+08:00
2,3,103.625520,1.289570,2026-06-20T09:10:14+08:00,1986,2026-06-20T09:10:17+08:00
3,4,103.626540,1.295260,2026-06-20T09:10:14+08:00,1986,2026-06-20T09:10:17+08:00
4,5,103.626540,1.295270,2026-06-20T09:10:14+08:00,1986,2026-06-20T09:10:17+08:00


## 10. Read the Raw CSV Back

This confirms that all three raw API results were saved.

If you run the download and save cells again, the number of rows should increase by 3.

In [27]:
history_df = pd.read_csv(raw_file)

print("Rows saved so far:", len(history_df))
history_df[["api_name", "status_code", "downloaded_at"]].tail(9)

Rows saved so far: 21


,api_name,status_code,downloaded_at
12,taxi_availability,200,2026-06-20T08:35:17+08:00
13,weather_forecast,200,2026-06-20T08:35:18+08:00
14,rainfall,200,2026-06-20T08:35:19+08:00
15,taxi_availability,200,2026-06-20T09:03:11+08:00
16,weather_forecast,200,2026-06-20T09:03:12+08:00
17,rainfall,200,2026-06-20T09:03:14+08:00
18,taxi_availability,200,2026-06-20T09:10:17+08:00
19,weather_forecast,200,2026-06-20T09:10:18+08:00
20,rainfall,200,2026-06-20T09:10:20+08:00


## 11. Stop Here

For the first project notebook, this is enough.

Students should now understand the basic loop:

```text
API key + API URL -> download JSON -> look at JSON -> save raw CSV -> save table-shaped CSV -> read CSV back
```

Next steps are separate practice:

- JSON practice: open `day_1_2JSON.ipynb`.
- CSV practice: open `day_1_3CSV.ipynb`.